# Chapter 2

In [8]:
import sys
import os
import torch

# Add project root so we can import the local qwen3.py directly
sys.path.insert(0, os.path.abspath(".."))
import qwen3 as Q

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")

Device : mps
PyTorch: 2.12.1


## 1. Download weights

`download_qwen3_small` fetches two files from HuggingFace (`rasbt/qwen3-from-scratch`) and saves them to the project root:
- `qwen3-0.6B-base.pth` — pretrained weights (~1.2 GB)
- `tokenizer-base.json` — BPE tokenizer config

Already-downloaded files are skipped automatically.

In [2]:
Q.download_qwen3_small(kind="base", out_dir="..")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


## 2. Tokenizer

`Qwen3Tokenizer` wraps a HuggingFace `tokenizers` BPE model and adds:
- Special-token handling (e.g. `<|im_start|>`, `<|endoftext|>`)
- An optional chat template wrapper (`apply_chat_template=True`)
- EOS token logic that differs between base and reasoning variants

In [3]:
tokenizer = Q.Qwen3Tokenizer("../tokenizer-base.json")

text = "The transformer architecture relies on self-attention."
ids  = tokenizer.encode(text)
back = tokenizer.decode(ids)

print(f"Original : {text!r}")
print(f"Token IDs ({len(ids)} tokens): {ids}")
print(f"Decoded  : {back!r}")
print(f"EOS token: {tokenizer.eos_token!r}  (id={tokenizer.eos_token_id})")

Original : 'The transformer architecture relies on self-attention.'
Token IDs (9 tokens): [785, 42578, 17646, 33644, 389, 656, 12, 53103, 13]
Decoded  : 'The transformer architecture relies on self-attention.'
EOS token: '<|endoftext|>'  (id=151643)


## 3. Model

`QWEN_CONFIG_06_B` holds the architecture hyperparameters. Key diff plain GPT-style LLM:
- **GQA** (`n_heads=16`, `n_kv_groups=8`): 8 KV heads shared across 16 query heads 
- **RoPE** (`rope_base=1_000_000`): positional encoding baked into attention, not a separate embedding
- **RMSNorm** instead of LayerNorm: no mean subtraction, slightly cheaper
- **SwiGLU** feed-forward: two parallel linear projections gated with SiLU, then projected down
- **`head_dim=128`** fixed: Q head dim is decoupled from `emb_dim / n_heads`

In [4]:
model = Q.Qwen3Model(Q.QWEN_CONFIG_06_B)

state_dict = torch.load("../qwen3-0.6B-base.pth", map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

total  = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total/1e6:.1f}M total")
print(f"dtype     : {next(model.parameters()).dtype}")
print(f"device    : {next(model.parameters()).device}")

Parameters: 751.6M total
dtype     : torch.bfloat16
device    : mps:0


## 4. Generate text

Two-phase autoregressive generation using the KV cache:
1. **Prefill** — process the full prompt in one forward pass; fills the cache for every layer
2. **Decode** — generate one token at a time; each step only computes attention over the new token against cached K/V

In [5]:
@torch.inference_mode()
def generate(model, tokenizer, prompt, max_new_tokens=100, top_k=1, temperature=1.0):
    token_ids    = tokenizer.encode(prompt)
    input_tensor = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)

    cache = Q.KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # prefill: process the whole prompt in one shot
    logits = model(input_tensor, cache=cache)  # (1, prompt_len, vocab_size)

    generated = list(token_ids)
    for _ in range(max_new_tokens):
        next_logits = logits[:, -1, :]   # prediction for the next position

        if temperature != 1.0:
            next_logits = next_logits / temperature

        if top_k > 1:
            topk_vals, topk_idx = torch.topk(next_logits, top_k)
            probs      = torch.softmax(topk_vals, dim=-1)
            next_token = topk_idx[0, torch.multinomial(probs[0], 1)].item()
        else:
            next_token = next_logits.argmax(dim=-1).item()  # greedy

        if next_token == tokenizer.eos_token_id:
            break

        generated.append(next_token)

        # decode: one token at a time — KV cache skips recomputing past keys/values
        next_input = torch.tensor([[next_token]], dtype=torch.long, device=device)
        logits     = model(next_input, cache=cache)  # (1, 1, vocab_size)

    return tokenizer.decode(generated)

In [7]:
prompt = "The key idea behind transformers is"
print(f"Prompt:\n{prompt}\n")
print("Output:")
print(generate(model, tokenizer, prompt, max_new_tokens=100, top_k=1))

Prompt:
The key idea behind transformers is

Output:
The key idea behind transformers is that the input and output of the transformer are the same. This is because the transformer is a linear operator, and linear operators preserve the structure of the input space. In other words, the output of the transformer is a linear combination of the input, and the input is a linear combination of the output. This is why the input and output of the transformer are the same.
The key idea behind transformers is that the input and output of the transformer are the same. This is because the transformer is a


## 5. Streaming generation

Same logic as `generate`, but `yield`s each new text chunk as soon as it's sampled instead of collecting everything first.

One subtlety: byte-level BPE tokens don't always align to clean character boundaries, so decoding a single token ID can produce garbled output. The safe approach is to decode the full `generated` list each step and yield only the *new suffix* — this handles multi-byte UTF-8 tokens correctly.

In [9]:
@torch.inference_mode()
def generate_stream(model, tokenizer, prompt, max_new_tokens=100, top_k=1, temperature=1.0):
    token_ids     = tokenizer.encode(prompt)
    input_tensor  = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)

    cache = Q.KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    logits = model(input_tensor, cache=cache)

    generated      = list(token_ids)
    decoded_so_far = tokenizer.decode(generated)

    for _ in range(max_new_tokens):
        next_logits = logits[:, -1, :]

        if temperature != 1.0:
            next_logits = next_logits / temperature

        if top_k > 1:
            topk_vals, topk_idx = torch.topk(next_logits, top_k)
            probs      = torch.softmax(topk_vals, dim=-1)
            next_token = topk_idx[0, torch.multinomial(probs[0], 1)].item()
        else:
            next_token = next_logits.argmax(dim=-1).item()

        if next_token == tokenizer.eos_token_id:
            break

        generated.append(next_token)

        new_decoded = tokenizer.decode(generated)
        yield new_decoded[len(decoded_so_far):]  # only the new text since last yield
        decoded_so_far = new_decoded

        next_input = torch.tensor([[next_token]], dtype=torch.long, device=device)
        logits     = model(next_input, cache=cache)

In [12]:
prompt = "Large language models are"
print(f"Prompt:\n{prompt}\n\nOutput:")
for chunk in generate_stream(model, tokenizer, prompt, max_new_tokens=100, top_k=1):
    print(chunk, end="", flush=True)
print()

Prompt:
Large language models are

Output:
 a type of artificial intelligence that can generate text based on a given input. They are trained on large amounts of text data and can be used to answer questions, generate creative content, and more. Large language models are becoming increasingly popular in various industries, including healthcare, finance, and education.
What is a large language model?
A large language model is a type of artificial intelligence that can generate text based on a given input. It is trained on large amounts of text data and can be used to answer


## 6. Benchmarking


In [13]:
import time
from utils import generate_stats

In [18]:
prompt = "Explain large language models in a single sentence."
input_token_ids = tokenizer.encode(prompt)

start_time = time.time()
output_chunks = []

for chunk in generate_stream(model, tokenizer, prompt, max_new_tokens=100, top_k=1):
    print(chunk, end="", flush=True)
    output_chunks.append(chunk)

end_time = time.time()

# encode the generated text to get the token count for stats
generated_text = "".join(output_chunks)
output_token_ids = torch.tensor(tokenizer.encode(generated_text), device=device)
generate_stats(output_token_ids, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Time: 1.78 sec
19 tokens/sec


![](1.png)

## 7. Using `torch.compile`

### What `torch.compile` does

`torch.compile` runs two passes over your model:

1. **TorchDynamo** — intercepts Python bytecode and traces the forward pass into a pure computation graph (an FX graph), stripping out Python-level control flow and overhead.

2. **TorchInductor** — takes that graph and generates optimized GPU kernels. The key win is **operator fusion**: instead of launching a separate kernel for each layer's `RMSNorm → attention → softmax → matmul → …`, multiple ops are fused into a single kernel. Fewer kernel launches → lower latency per token.

On **MPS** (Apple Silicon) Inductor uses the Metal backend. On **CUDA** it generates custom Triton kernels.

### Why the first run is slow

The very first forward pass after `torch.compile` triggers the actual compilation — Dynamo traces the graph, Inductor generates and compiles the kernels. This warm-up can take 30–60 seconds. Every subsequent run uses the cached compiled kernels and is fast.

### The `allow_unspec_int_on_nn_module` flag

`Qwen3Model` tracks `self.current_pos` (an integer on an `nn.Module`) and increments it each decode step. Without the flag, Dynamo would treat every new value of `current_pos` as a new input shape and retrigger recompilation on *every token* — making it slower than eager. The flag tells Dynamo to compile once and handle varying integer values dynamically.

In [19]:
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    # Prevents recompilation every decode step when current_pos changes
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)
print("Model compiled — first generate call will be slow (warm-up)")

Model compiled — first generate call will be slow (warm-up)


In [20]:
prompt = "Explain large language models in a single sentence."

for i in range(3):
    start_time = time.time()
    output_chunks = []

    for chunk in generate_stream(model_compiled, tokenizer, prompt, max_new_tokens=100, top_k=1):
        print(chunk, end="", flush=True)
        output_chunks.append(chunk)

    end_time = time.time()

    label = "Warm-up run (compilation happening)" if i == 0 else f"Timed run {i}"
    print(f"\n\n--- {label} ---")
    output_token_ids = torch.tensor(tokenizer.encode("".join(output_chunks)), device=device)
    generate_stats(output_token_ids, tokenizer, start_time, end_time)
    print()

W0704 23:49:53.993000 82419 .venv/lib/python3.14/site-packages/torch/_inductor/utils.py:1717] [0/0] Not enough SMs to use max_autotune_gemm mode


 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

--- Warm-up run (compilation happening) ---


Time: 94.60 sec
0 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

--- Timed run 1 ---


Time: 1.36 sec
30 tokens/sec

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

--- Timed run 2 ---


Time: 1.34 sec
30 tokens/sec

